# Fase B: Análisis Offline de Biodiversidad desde MySQL 8

Este cuaderno lee los datos almacenados localmente en la base de datos `gbif_biodiversidad` con la sesión ORM (`get_session` + `BiodiversidadRepository`), sin realizar peticiones externas a la API de GBIF.

In [1]:
import importlib
import sys
from pathlib import Path
import pandas as pd

raiz = Path.cwd()
if not (raiz / "src").is_dir():
    raiz = raiz.parent
ruta_src = raiz / "src"
ruta_src = ruta_src.resolve()
if str(ruta_src) not in sys.path:
    sys.path.insert(0, str(ruta_src))

import repository
import consulta_gbif

importlib.reload(repository)
importlib.reload(consulta_gbif)

from database import get_session
from repository import BiodiversidadRepository
from consulta_gbif import (
    cargar_ocurrencias,
    consultar_campos_raw_json,
    obtener_resumen_general,
    resumen_territorial_por_comunidad,
    top_datasets_publicadores,
)

with get_session() as session:
    conteos_iniciales = BiodiversidadRepository(session).resumen_conteo_tablas()

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)
print(f"ORM listo. src en: {ruta_src}")
print("Conteos:", conteos_iniciales)

ORM listo. src en: C:\Users\chuwi\Documents\biodiversidad\src
Conteos: {'taxones': 3, 'organizaciones': 14, 'datasets': 19, 'ocurrencias': 4337}


## 1. Estado de la base de datos local

In [2]:
df_resumen_bd = obtener_resumen_general()
df_resumen_bd

,tabla,total
0,taxones,3
1,organizaciones,14
2,datasets,19
3,ocurrencias,4337


## 2. Carga de ocurrencias (`cargar_ocurrencias`)

Carga los registros de España (`ES`) y Portugal (`PT`) filtrando únicamente el lobo silvestre (`solo_lobo_silvestre=True`) o el conjunto completo (`solo_lobo_silvestre=False`).

In [3]:
# Carga completa (ES + PT)
df_todos = cargar_ocurrencias(paises=("ES", "PT"), solo_lobo_silvestre=False)

# Carga filtrada: solo lobo silvestre (excluye Canis lupus familiaris)
df = cargar_ocurrencias(paises=("ES", "PT"), solo_lobo_silvestre=True, anio_min=2000)

print(f"Total registros en BD (ES + PT): {len(df_todos):,}")
print(f"Registros de lobo silvestre (>= 2000): {len(df):,}")

df[
    [
        "key",
        "countryCode",
        "year",
        "scientificName",
        "comunidad_gadm",
        "provincia_gadm",
        "basisOfRecord",
        "publishing_org_title",
        "gbif_url",
    ]
].head(10)

Total registros en BD (ES + PT): 4,337
Registros de lobo silvestre (>= 2000): 880


,key,countryCode,year,scientificName,comunidad_gadm,provincia_gadm,basisOfRecord,publishing_org_title,gbif_url
0,6907542303,ES,2026,"Canis lupus Linnaeus, 1758",Castilla y León,Zamora,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6907542303
1,6885782502,PT,2026,"Canis lupus Linnaeus, 1758",Braga,Terras de Bouro,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6885782502
2,6885119513,ES,2026,"Canis lupus Linnaeus, 1758",Principado de Asturias,Asturias,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6885119513
3,6871713228,ES,2026,"Canis lupus Linnaeus, 1758",NaN,NaN,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6871713228
4,6871646127,PT,2026,"Canis lupus Linnaeus, 1758",Castilla y León,Zamora,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6871646127
5,6871558851,ES,2026,"Canis lupus Linnaeus, 1758",Principado de Asturias,Asturias,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6871558851
6,6871537048,PT,2026,"Canis lupus Linnaeus, 1758",Galicia,Ourense,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6871537048
7,6871476663,ES,2026,"Canis lupus Linnaeus, 1758",Castilla y León,León,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6871476663
8,6871167428,ES,2026,"Canis lupus Linnaeus, 1758",Castilla y León,Zamora,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6871167428
9,6871055072,ES,2026,"Canis lupus Linnaeus, 1758",Principado de Asturias,Asturias,HUMAN_OBSERVATION,iNaturalist.org,https://www.gbif.org/occurrence/6871055072


## 3. Distribución territorial y fuentes publicadoras

In [4]:
df_territorial_es = resumen_territorial_por_comunidad(pais="ES")
display(df_territorial_es.head(15))

df_top_datasets = top_datasets_publicadores(limite=10)
display(df_top_datasets)

,region,tipo_evidencia,total_registros,desde_anio,hasta_anio
0,Andalucía,HUMAN_OBSERVATION,658,2023,2026
1,Galicia,HUMAN_OBSERVATION,262,2024,2026
2,Castilla y León,HUMAN_OBSERVATION,228,2024,2026
3,Castilla-La Mancha,HUMAN_OBSERVATION,142,2023,2026
4,Cataluña,HUMAN_OBSERVATION,128,2024,2026
5,Aragón,HUMAN_OBSERVATION,100,2024,2026
6,Principado de Asturias,HUMAN_OBSERVATION,100,2024,2026
7,Comunidad Foral de Navarra,HUMAN_OBSERVATION,94,2024,2026
8,Extremadura,HUMAN_OBSERVATION,74,2024,2026
9,Comunidad Valenciana,HUMAN_OBSERVATION,68,2024,2026


,dataset,organizacion,pais_org,licencia,registros_en_bd
0,iNaturalist Research-grade Observations,iNaturalist.org,US,http://creativecommons.org/licenses/by-nc/4.0/legalcode,1806
1,"Observation.org, Nature data from around the World",Observation.org,NL,http://creativecommons.org/licenses/by-nc/4.0/legalcode,852
2,Long-term monitoring of mammal communities in the Peneda...,Biodiversity Data Journal,BG,http://creativecommons.org/licenses/by/4.0/legalcode,483
3,Mammals – Camera Trap Surveys – Spain – 2014–2026 – MOMA...,Sociedad Ibérica para la Conservación y Estudio de los M...,ES,http://creativecommons.org/licenses/by/4.0/legalcode,220
4,Wildlife inventory in the Azores using camera trapping,Universidade dos Açores,PT,http://creativecommons.org/licenses/by/4.0/legalcode,210
5,Mammal records for Europe via the iMammalia app,Biological Records Centre,GB,http://creativecommons.org/licenses/by/4.0/legalcode,149
6,EDP Baixo Sabor: Construction and Reservoir Filling Phas...,EDP,PT,http://creativecommons.org/licenses/by/4.0/legalcode,130
7,Global Roadkill Data: a dataset on terrestrial vertebrat...,CIBIO (Research Center in Biodiversity and Genetic Resou...,PT,http://creativecommons.org/licenses/by/4.0/legalcode,101
8,EDP Baixo Sabor: Construction and Reservoir Filling Phas...,EDP,PT,http://creativecommons.org/licenses/by/4.0/legalcode,101
9,Mammals – Transect Sign Surveys – Spain – 2022–2025 – MO...,Sociedad Ibérica para la Conservación y Estudio de los M...,ES,http://creativecommons.org/licenses/by/4.0/legalcode,76


## 4. Consulta dinámica sobre `raw_json` (multimedia y hábitat)

In [5]:
df_multimedia = consultar_campos_raw_json(limite=15)
print(f"Registros encontrados con multimedia o hábitat en JSON: {len(df_multimedia)}")
df_multimedia

Registros encontrados con multimedia o hábitat en JSON: 15


,occurrence_key,country_code,year,scientific_name,habitat_json,comentarios_evento,num_archivos_media,url_primera_foto,gbif_url
0,6919484564,ES,2026,"Lynx pardinus (Temminck, 1827)",None,None,1,https://inaturalist-open-data.s3.amazonaws.com/photos/76...,https://www.gbif.org/occurrence/6919484564
1,6919470686,PT,2026,"Vulpes vulpes (Linnaeus, 1758)",None,None,1,https://inaturalist-open-data.s3.amazonaws.com/photos/76...,https://www.gbif.org/occurrence/6919470686
2,6919428488,PT,2026,"Vulpes vulpes (Linnaeus, 1758)",None,None,2,https://inaturalist-open-data.s3.amazonaws.com/photos/76...,https://www.gbif.org/occurrence/6919428488
3,6919362825,ES,2026,"Vulpes vulpes (Linnaeus, 1758)",None,None,1,https://inaturalist-open-data.s3.amazonaws.com/photos/76...,https://www.gbif.org/occurrence/6919362825
4,6918900791,ES,2026,"Vulpes vulpes (Linnaeus, 1758)",None,None,1,https://inaturalist-open-data.s3.amazonaws.com/photos/76...,https://www.gbif.org/occurrence/6918900791
5,6918491744,PT,2026,"Vulpes vulpes (Linnaeus, 1758)",None,None,3,https://inaturalist-open-data.s3.amazonaws.com/photos/76...,https://www.gbif.org/occurrence/6918491744
6,6907819609,PT,2026,"Vulpes vulpes (Linnaeus, 1758)",None,None,1,https://inaturalist-open-data.s3.amazonaws.com/photos/76...,https://www.gbif.org/occurrence/6907819609
7,6907760762,PT,2026,"Vulpes vulpes (Linnaeus, 1758)",None,None,2,https://inaturalist-open-data.s3.amazonaws.com/photos/76...,https://www.gbif.org/occurrence/6907760762
8,6907642691,PT,2026,"Vulpes vulpes (Linnaeus, 1758)",None,None,2,https://inaturalist-open-data.s3.amazonaws.com/photos/76...,https://www.gbif.org/occurrence/6907642691
9,6907542303,ES,2026,"Canis lupus Linnaeus, 1758",None,None,4,https://inaturalist-open-data.s3.amazonaws.com/photos/75...,https://www.gbif.org/occurrence/6907542303


## 5. Mapa interactivo rápido con Folium (sin API Key)

In [6]:
from pathlib import Path
import sys
import folium
from folium.plugins import MarkerCluster

_raiz_enlaces = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if str(_raiz_enlaces / "src") not in sys.path:
    sys.path.insert(0, str(_raiz_enlaces / "src"))
from mapa_enlaces import activar_enlaces_externos

# 1. Crear carpeta de salida si no existe
carpeta_salida = (Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent) / "salidas_gbif_multiespecie"
carpeta_salida.mkdir(parents=True, exist_ok=True)
ruta_mapa_html = carpeta_salida / "mapa_ocurrencias_gbif.html"

# 2. Filtrar registros con coordenadas válidas
df_mapa = df.dropna(subset=["decimalLatitude", "decimalLongitude"]).copy()

mapa = folium.Map(
    location=[41.8, -5.5],
    zoom_start=6,
    tiles="OpenStreetMap"
)

# Capas base gratuitas sin API key
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satélite (Esri)",
    overlay=False,
    control=True,
).add_to(mapa)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Street Map",
    name="Callejero (Esri)",
    overlay=False,
    control=True,
).add_to(mapa)

cluster = MarkerCluster(name="Ocurrencias Canis lupus").add_to(mapa)

for _, fila in df_mapa.iterrows():
    url_img = fila.get("url_imagen")
    tiene_imagen = isinstance(url_img, str) and url_img.startswith("http")

    bloque_imagen = ""
    if tiene_imagen:
        bloque_imagen = (
            f"<div style='margin:6px 0;'>"
            f"<a href='{url_img}' target='_blank' rel='noopener noreferrer'>"
            f"<img src='{url_img}' alt='Foto avistamiento' "
            f"style='width:220px; max-height:150px; object-fit:cover; border-radius:4px; display:block; margin-bottom:4px;'>"
            f"Ver imagen original ({int(fila.get('num_media', 1))} media)</a>"
            f"</div>"
        )

    popup_html = (
        f"<div style='font-family: Arial, sans-serif; font-size: 12px;'>"
        f"<b>{fila['scientificName']}</b><br>"
        f"País: {fila['countryCode']} | Año: {fila['year']}<br>"
        f"Región: {fila['comunidad_gadm'] or 'N/D'}<br>"
        f"Evidencia: {fila['basisOfRecord']}<br>"
        f"Fuente: {fila['publishing_org_title'] or 'N/D'}<br>"
        f"{bloque_imagen}"
        f"<a href='{fila['gbif_url']}' target='_blank' rel='noopener noreferrer'>Ver ficha completa en GBIF</a>"
        f"</div>"
    )

    folium.CircleMarker(
        location=[fila["decimalLatitude"], fila["decimalLongitude"]],
        radius=6 if tiene_imagen else 5,
        color="#d00000" if tiene_imagen else "#1b4332",
        fill=True,
        fill_color="#e63946" if tiene_imagen else "#2d6a4f",
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=260),
    ).add_to(cluster)

folium.LayerControl().add_to(mapa)

# 3. Guardar mapa en salidas_gbif_multiespecie/
activar_enlaces_externos(mapa)
mapa.save(str(ruta_mapa_html))
print(f"Mapa guardado correctamente en: {ruta_mapa_html.resolve()}")
print(f"Registros con imagen directa: {df_mapa['url_imagen'].notna().sum()} de {len(df_mapa)}")

mapa




Mapa guardado correctamente en: C:\Users\chuwi\Documents\biodiversidad\salidas_gbif_multiespecie\mapa_ocurrencias_gbif.html
Registros con imagen directa: 524 de 880
